# Supply Chain Graph Analytics Visualization
Interactive network graphs of seller-warehouse relationships, risk levels, communities, and PageRank centrality.

In [ ]:
%%sql -r sellers_df
SELECT
    s.NODEID, s.SELLER_NAME, s.PLATFORM, s.CITY,
    s.RETURN_RATE, s.FRAUD_FLAG,
    r.PAGERANK_SCORE, r.LOUVAIN_COMMUNITY, r.WCC_ENTITY,
    r.RISK_SCORE, r.RISK_LEVEL
FROM SUPPLY_CHAIN_DB.PUBLIC.SELLERS s
LEFT JOIN SUPPLY_CHAIN_DB.PUBLIC.SELLER_RISK_MASTER r ON s.NODEID = r.NODEID
GROUP BY ALL

In [ ]:
%%sql -r warehouses_df
SELECT NODEID FROM SUPPLY_CHAIN_DB.PUBLIC.WAREHOUSES_GRAPH

In [ ]:
%%sql -r edges_df
SELECT
    SOURCENODEID, TARGETNODEID,
    SUM(ORDER_VALUE) AS TOTAL_ORDER_VALUE,
    COUNT(*) AS ORDER_COUNT
FROM SUPPLY_CHAIN_DB.PUBLIC.ORDERS_GRAPH
GROUP BY SOURCENODEID, TARGETNODEID

In [ ]:
import networkx as nx
import plotly.graph_objects as go
import numpy as np

G = nx.Graph()

for _, row in sellers_df.iterrows():
    G.add_node(row['NODEID'], node_type='seller', label=row['SELLER_NAME'],
               city=row['CITY'], risk_level=row['RISK_LEVEL'],
               risk_score=row['RISK_SCORE'], pagerank=row['PAGERANK_SCORE'],
               community=row['LOUVAIN_COMMUNITY'], platform=row['PLATFORM'])

for _, row in warehouses_df.iterrows():
    G.add_node(row['NODEID'], node_type='warehouse', label=f"WH-{row['NODEID']}")

for _, row in edges_df.iterrows():
    G.add_edge(row['SOURCENODEID'], row['TARGETNODEID'],
               weight=row['TOTAL_ORDER_VALUE'], order_count=row['ORDER_COUNT'])

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
pos = nx.spring_layout(G, seed=42, k=0.5, iterations=50)

seller_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'seller']
wh_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'warehouse']

risk_color_map = {'HIGH': '#FF4444', 'MEDIUM': '#FFA500', 'LOW': '#44BB44'}

edge_x, edge_y = [], []
for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.3, color='#cccccc'), hoverinfo='none', name='Orders')

seller_x = [pos[n][0] for n in seller_nodes]
seller_y = [pos[n][1] for n in seller_nodes]
seller_colors = [risk_color_map.get(G.nodes[n].get('risk_level', 'LOW'), '#44BB44') for n in seller_nodes]
seller_sizes = [max(6, (G.nodes[n].get('pagerank') or 0) * 80 + 6) for n in seller_nodes]
seller_text = [
    f"<b>{G.nodes[n].get('label','')}</b><br>"
    f"City: {G.nodes[n].get('city','')}<br>"
    f"Risk: {G.nodes[n].get('risk_level','')} ({G.nodes[n].get('risk_score','')})<br>"
    f"PageRank: {G.nodes[n].get('pagerank','')}<br>"
    f"Community: {G.nodes[n].get('community','')}<br>"
    f"Platform: {G.nodes[n].get('platform','')}"
    for n in seller_nodes
]

seller_trace = go.Scatter(x=seller_x, y=seller_y, mode='markers',
    marker=dict(size=seller_sizes, color=seller_colors, line=dict(width=0.5, color='white')),
    text=seller_text, hoverinfo='text', name='Sellers')

wh_x = [pos[n][0] for n in wh_nodes]
wh_y = [pos[n][1] for n in wh_nodes]
wh_text = [f"<b>Warehouse {n}</b><br>Connections: {G.degree(n)}" for n in wh_nodes]

wh_trace = go.Scatter(x=wh_x, y=wh_y, mode='markers',
    marker=dict(size=14, color='#4488FF', symbol='diamond',
                line=dict(width=1, color='white')),
    text=wh_text, hoverinfo='text', name='Warehouses')

fig = go.Figure(data=[edge_trace, seller_trace, wh_trace],
    layout=go.Layout(
        title='Supply Chain Network: Sellers → Warehouses<br><sup>Node size = PageRank | Color = Risk Level (Red=High, Orange=Medium, Green=Low) | Diamond = Warehouse</sup>',
        showlegend=True, hovermode='closest',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        width=1000, height=700,
        margin=dict(l=20, r=20, t=60, b=20),
        plot_bgcolor='white'
    ))
fig.show()

In [ ]:
communities = {}
for n in seller_nodes:
    c = G.nodes[n].get('community')
    if c is not None:
        communities.setdefault(int(c), []).append(n)

import random
random.seed(42)
palette = [f'hsl({int(i * 360 / max(len(communities),1))}, 70%, 50%)' for i in range(len(communities))]

fig2 = go.Figure()
fig2.add_trace(edge_trace)

for idx, (comm_id, members) in enumerate(sorted(communities.items())):
    cx = [pos[n][0] for n in members]
    cy = [pos[n][1] for n in members]
    ct = [f"<b>{G.nodes[n].get('label','')}</b><br>Community: {comm_id}" for n in members]
    fig2.add_trace(go.Scatter(x=cx, y=cy, mode='markers',
        marker=dict(size=8, color=palette[idx % len(palette)]),
        text=ct, hoverinfo='text', name=f'Community {comm_id}'))

fig2.add_trace(go.Scatter(x=wh_x, y=wh_y, mode='markers',
    marker=dict(size=12, color='#4488FF', symbol='diamond'),
    text=wh_text, hoverinfo='text', name='Warehouses'))

fig2.update_layout(
    title='Louvain Community Detection<br><sup>Each color = one community cluster</sup>',
    showlegend=True, hovermode='closest',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    width=1000, height=700, plot_bgcolor='white',
    margin=dict(l=20, r=20, t=60, b=20))
fig2.show()

In [ ]:
import plotly.express as px
import pandas as pd

risk_data = sellers_df[['RISK_LEVEL', 'RISK_SCORE', 'PAGERANK_SCORE', 'LOUVAIN_COMMUNITY', 'CITY', 'SELLER_NAME', 'PLATFORM']].dropna(subset=['RISK_LEVEL'])

fig3 = px.scatter(risk_data, x='PAGERANK_SCORE', y='RISK_SCORE',
    color='RISK_LEVEL', hover_data=['SELLER_NAME', 'CITY', 'PLATFORM', 'LOUVAIN_COMMUNITY'],
    color_discrete_map={'HIGH': '#FF4444', 'MEDIUM': '#FFA500', 'LOW': '#44BB44'},
    title='PageRank vs Risk Score by Seller',
    labels={'PAGERANK_SCORE': 'PageRank Centrality', 'RISK_SCORE': 'Risk Score'})
fig3.update_layout(width=900, height=500)
fig3.show()

risk_city = risk_data.groupby(['CITY', 'RISK_LEVEL']).size().reset_index(name='COUNT')
fig4 = px.bar(risk_city, x='CITY', y='COUNT', color='RISK_LEVEL',
    color_discrete_map={'HIGH': '#FF4444', 'MEDIUM': '#FFA500', 'LOW': '#44BB44'},
    title='Risk Distribution by City', barmode='stack')
fig4.update_layout(width=900, height=400)
fig4.show()